# Przykład 2: Q-learning — dojazd do celu w 1D

Robot porusza się po osi. Obserwacją jest błąd `e = x_cel - x_robot`, a stanem — przedział, do którego ten błąd należy. Akcją jest prędkość wybrana ze skończonego zbioru.

Pozycja zmienia się o `v · Δt`. Sukces oznacza `|e| < TOLERANCE`; za sukces agent dostaje `+10`, a w pozostałych krokach karę `-|e| - 0.01`.

Algorytm Q-learningu jest taki sam jak w przykładzie z korytarzem. Tutaj dodatkowo dyskretyzujemy ciągłą obserwację. To uproszczony dojazd w 1D — bez modelowania bezwładności i hamowania. Uruchom komórki po kolei i obserwuj wyniki.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Parametry

In [ ]:
# --- środowisko ---
X_MIN, X_MAX = -5.0, 5.0     # zakres pozycji robota i celu [m]
DELTA_T = 0.2                # czas między decyzjami agenta [s]
TOLERANCE = 0.1              # sukces, gdy |e| < TOLERANCE [m]
MIN_START_DISTANCE = 0.5     # minimalna początkowa odległość od celu, > TOLERANCE [m]
MAX_STEPS = 300              # limit kroków w jednym epizodzie [kroki]

# --- akcje ---
SPEEDS = np.array([-1.0, -0.5, 0.0, 0.5, 1.0])   # dostępne prędkości [m/s]

# --- dyskretyzacja stanu ---
E_MIN, E_MAX = X_MIN - X_MAX, X_MAX - X_MIN      # zakres błędu e: [-10, 10] [m]
N_BINS = 100                                     # liczba przedziałów błędu [-]

# --- nagrody ---
REWARD_GOAL = 10.0           # nagroda za osiągnięcie celu [-]
STEP_PENALTY = 0.01          # stała kara za każdy krok [-]

# --- Q-learning ---
N_EPISODES = 3000            # liczba epizodów uczenia [-]
ALPHA = 0.2                  # współczynnik uczenia, (0, 1] [-]
GAMMA = 0.95                 # współczynnik dyskontowania, (0, 1] [-]

# --- strategia epsilon-zachłanna ---
EPSILON_START = 1.0          # epsilon w pierwszym epizodzie, [0, 1] [-]
EPSILON_END = 0.05           # epsilon w ostatnim epizodzie, [0, 1] [-]
EPSILON_MODE = "exp"         # "const", "linear" lub "exp"

SEED = 0                     # ziarno generatora losowego (powtarzalność wyników)

## Środowisko

Środowisko przesuwa robota i zwraca ciągły błąd położenia.

In [ ]:
class ParkingEnv:
    """Dojazd do celu na osi. Obserwacją jest błąd położenia e = x_cel - x_robot [m]."""

    def __init__(self, x_min, x_max, delta_t, tolerance, min_start_distance, speeds):
        self.x_min, self.x_max = x_min, x_max
        self.delta_t = delta_t                       # [s]
        self.tolerance = tolerance                   # [m]
        self.min_start_distance = min_start_distance # [m]
        self.speeds = np.asarray(speeds, dtype=float)  # [m/s]
        self.x_robot = 0.0                           # pozycja robota [m]
        self.x_goal = 0.0                            # pozycja celu [m]

    def reset(self, rng, x_robot=None, x_goal=None):
        """Losuje pozycje robota i celu (lub przyjmuje zadane). Zwraca e [m]."""
        if x_robot is not None and x_goal is not None:
            self.x_robot, self.x_goal = float(x_robot), float(x_goal)
        else:
            while True:   # losujemy do skutku, by start nie był już "w celu"
                self.x_robot = rng.uniform(self.x_min, self.x_max)
                self.x_goal = rng.uniform(self.x_min, self.x_max)
                if abs(self.x_goal - self.x_robot) >= self.min_start_distance:
                    break
        return self.x_goal - self.x_robot

    def step(self, action):
        """Wykonuje akcję (indeks prędkości). Zwraca (e, nagroda, czy_terminalny)."""
        if not 0 <= action < len(self.speeds):
            raise ValueError(f"Niepoprawny numer akcji: {action}")
        v = self.speeds[action]                                   # [m/s]
        self.x_robot += v * self.delta_t                          # [m]
        e = self.x_goal - self.x_robot                            # [m]
        if abs(e) < self.tolerance:
            return e, REWARD_GOAL, True
        return e, -abs(e) - STEP_PENALTY, False


def make_env(delta_t=None, speeds=None):
    return ParkingEnv(X_MIN, X_MAX, DELTA_T if delta_t is None else delta_t, TOLERANCE,
                      MIN_START_DISTANCE, SPEEDS if speeds is None else speeds)


def draw_axis(ax, x_robot, x_goal, title=""):
    """Rysuje oś z robotem (niebieski) i celem (czerwona gwiazdka)."""
    ax.axhline(0, color="gray", lw=1)
    ax.plot([x_goal], [0], "r*", ms=18, label="cel")
    ax.plot([x_robot], [0], "bs", ms=14, label="robot")
    ax.set(xlim=(X_MIN - 0.5, X_MAX + 0.5), ylim=(-1, 1), xlabel="x [m]", title=title)
    ax.set_yticks([])
    ax.legend(loc="upper right")


fig, ax = plt.subplots(figsize=(8, 1.8))
env_demo = make_env()
e0 = env_demo.reset(np.random.default_rng(SEED))
draw_axis(ax, env_demo.x_robot, env_demo.x_goal, f"Przykładowy początek epizodu: e = {e0:+.2f} m")
plt.show()

## Dyskretyzacja stanu

Zakres błędu dzielimy na `N_BINS` przedziałów. Każdy przedział odpowiada jednemu wierszowi tablicy Q; wartości poza zakresem trafiają do skrajnych przedziałów.

Przy domyślnym zakresie `[-10, 10]` i `N_BINS=100` szerokość binu wynosi 0,2 m, podczas gdy tolerancja sukcesu to 0,1 m. Reprezentacja stanu jest więc przybliżona w okolicy zera; środowisko niezależnie sprawdza warunek sukcesu.

In [ ]:
def discretize_error(e, n_bins=None, e_min=None, e_max=None):
    """Zamienia ciągłe e [m] na indeks przedziału 0 … n_bins-1."""
    n_bins = N_BINS if n_bins is None else n_bins
    e_min = E_MIN if e_min is None else e_min
    e_max = E_MAX if e_max is None else e_max
    if not np.isfinite(e):
        raise ValueError("Obserwacja musi być liczbą skończoną")
    width = (e_max - e_min) / n_bins
    index = int(np.floor((e - e_min) / width))
    return int(np.clip(index, 0, n_bins - 1))                   # przycięcie do skrajnych przedziałów


def bin_edges(index, n_bins=None, e_min=None, e_max=None):
    """Granice [dolna, górna) przedziału o danym indeksie [m]."""
    n_bins = N_BINS if n_bins is None else n_bins
    e_min = E_MIN if e_min is None else e_min
    e_max = E_MAX if e_max is None else e_max
    width = (e_max - e_min) / n_bins
    return e_min + index * width, e_min + (index + 1) * width


print(f"Szerokość przedziału: {(E_MAX - E_MIN) / N_BINS:.2f} m, liczba stanów: {N_BINS}\n")
print("   e [m] | indeks stanu | granice przedziału [m]")
for e in [-12.0, -10.0, -3.14, -0.25, -0.05, 0.0, 0.05, 0.25, 3.14, 9.99, 15.0]:
    s = discretize_error(e)
    lo, hi = bin_edges(s)
    print(f"{e:8.2f} | {s:12d} | [{lo:+.2f}, {hi:+.2f})")

### Przedziały błędu

Wartości w jednym przedziale są dla agenta tym samym stanem. Zielony pas oznacza tolerancję sukcesu.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2.2))
edges = np.linspace(E_MIN, E_MAX, N_BINS + 1)
for edge in edges:
    ax.axvline(edge, color="gray", lw=0.4)
ax.axvspan(-TOLERANCE, TOLERANCE, color="green", alpha=0.3, label=f"|e| < {TOLERANCE} m (sukces)")
sample = np.array([-7.3, -2.05, 0.37, 4.61])
ax.plot(sample, np.zeros_like(sample), "ro", label="przykładowe e")
for e in sample:
    ax.annotate(f"stan {discretize_error(e)}", (e, 0), xytext=(0, 12), textcoords="offset points",
                ha="center")
ax.set(xlabel="błąd e = x_cel - x_robot [m]", yticks=[], title="Dyskretyzacja błędu położenia")
ax.legend(loc="lower center", ncol=2)
plt.show()

## Q-learning

Wybór akcji i aktualizacja tablicy Q działają tak samo jak w przykładzie z korytarzem. Zmienna obserwacja jest najpierw zamieniana na stan dyskretny.

In [ ]:
def choose_action(q_table, state, epsilon, rng):
    """Strategia epsilon-zachłanna z losowym rozstrzyganiem remisów."""
    n_actions = q_table.shape[1]
    if rng.random() < epsilon:
        return int(rng.integers(n_actions))
    best = np.flatnonzero(q_table[state] == q_table[state].max())
    return int(rng.choice(best))


def update_q_table(q_table, state, action, reward, next_state, terminal, alpha, gamma):
    """Jedna aktualizacja Q-learningu dla przejścia (s, a, r, s')."""
    if terminal:
        target = reward                                           # brak następnego kroku
    else:
        target = reward + gamma * np.max(q_table[next_state])
    q_table[state, action] += alpha * (target - q_table[state, action])


def epsilon_for_episode(episode, n_episodes, eps_start, eps_end, mode):
    """Wartość epsilon w danym epizodzie (numerowanym od 0)."""
    if mode == "const":
        return eps_start
    progress = episode / max(1, n_episodes - 1)
    if mode == "linear":
        return eps_start + (eps_end - eps_start) * progress
    if mode == "exp":
        return eps_start * (eps_end / eps_start) ** progress
    raise ValueError(f"Nieznany tryb epsilon: {mode}")


def train(env, encode_state, n_states, n_actions, n_episodes, alpha, gamma,
          eps_start, eps_end, eps_mode, max_steps, seed):
    """Uczy agenta. Zwraca tablicę Q i historię uczenia (po jednej wartości na epizod)."""
    rng = np.random.default_rng(seed)
    q_table = np.zeros((n_states, n_actions))
    history = {"return": [], "steps": [], "success": [], "epsilon": []}

    for episode in range(n_episodes):
        epsilon = epsilon_for_episode(episode, n_episodes, eps_start, eps_end, eps_mode)
        state = encode_state(env.reset(rng))
        total_reward, terminal, steps = 0.0, False, 0

        while not terminal and steps < max_steps:
            action = choose_action(q_table, state, epsilon, rng)
            observation, reward, terminal = env.step(action)
            next_state = encode_state(observation)
            # Limit kroków NIE jest stanem terminalnym, więc nadal liczymy max Q(s', .).
            update_q_table(q_table, state, action, reward, next_state, terminal, alpha, gamma)
            state = next_state
            total_reward += reward
            steps += 1

        history["return"].append(total_reward)
        history["steps"].append(steps)
        history["success"].append(terminal)
        history["epsilon"].append(epsilon)
    return q_table, history


def run_greedy_episode(env, encode_state, q_table, max_steps, rng, x_robot=None, x_goal=None):
    """Epizod bez eksploracji (epsilon = 0). Zwraca ślad {t, x, x_goal, e, v, r} i sukces."""
    e = env.reset(rng, x_robot, x_goal)
    state = encode_state(e)
    trace = dict(t=[0.0], x=[env.x_robot], x_goal=[env.x_goal], e=[e], v=[], r=[])
    terminal = False
    for k in range(1, max_steps + 1):
        action = choose_action(q_table, state, 0.0, rng)
        e, reward, terminal = env.step(action)
        state = encode_state(e)
        trace["t"].append(k * env.delta_t)
        trace["x"].append(env.x_robot)
        trace["x_goal"].append(env.x_goal)
        trace["e"].append(e)
        trace["v"].append(float(env.speeds[action]))
        trace["r"].append(reward)
        if terminal:
            break
    return trace, terminal


def success_rate(env, encode_state, q_table, n_episodes, max_steps, seed):
    """Odsetek udanych epizodów w trybie zachłannym (epsilon = 0) i średnia liczba kroków."""
    rng = np.random.default_rng(seed)
    results = [run_greedy_episode(env, encode_state, q_table, max_steps, rng) for _ in range(n_episodes)]
    ok = np.array([r[1] for r in results])
    steps = np.array([len(r[0]["v"]) for r in results])
    return ok.mean(), (steps[ok].mean() if ok.any() else float("nan"))

## Początkowa tablica Q

Wiersze odpowiadają przedziałom błędu, a kolumny — prędkościom. Początkowe wartości wynoszą zero.

In [ ]:
env = make_env()
q_init = np.zeros((N_BINS, len(SPEEDS)))
print(f"Tablica Q: {q_init.shape[0]} stanów × {q_init.shape[1]} akcji = {q_init.size} wartości")
print("Akcje (prędkości [m/s]):", SPEEDS)
print("Pierwsze 3 wiersze:\n", q_init[:3])

## Trening

In [ ]:
encode = discretize_error
q_table, history = train(env, encode, N_BINS, len(SPEEDS), N_EPISODES, ALPHA, GAMMA,
                         EPSILON_START, EPSILON_END, EPSILON_MODE, MAX_STEPS, SEED)
print(f"Skuteczność w ostatnich 200 epizodach (z eksploracją): {np.mean(history['success'][-200:]):.0%}")
print(f"Odwiedzone stany: {int(np.sum(np.any(q_table != 0, axis=1)))} z {N_BINS}")

### Obserwuj
- Czy odsetek sukcesów rośnie, a czas dojazdu maleje?
- Wyniki różnią się między epizodami, bo pozycje startowe są losowe.

In [ ]:
def moving_average(values, window=50):
    values = np.asarray(values, dtype=float)
    if len(values) < window:
        return np.arange(len(values)), values
    ma = np.convolve(values, np.ones(window) / window, mode="valid")
    return np.arange(len(values) - len(ma), len(values)), ma


fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(history["return"], alpha=0.25, label="pojedynczy epizod")
axes[0].plot(*moving_average(history["return"]), label="średnia krocząca (50)")
axes[0].set(xlabel="numer epizodu [-]", ylabel="suma nagród [-]", title="Suma nagród w epizodzie")
axes[0].legend()

time_to_goal = np.array(history["steps"]) * DELTA_T
axes[1].plot(time_to_goal, alpha=0.25, label="pojedynczy epizod")
axes[1].plot(*moving_average(time_to_goal), label="średnia krocząca (50)")
axes[1].set(xlabel="numer epizodu [-]", ylabel="czas epizodu [s]", title="Czas do końca epizodu")
axes[1].legend()

axes[2].plot(*moving_average(np.array(history["success"], dtype=float)), label="odsetek sukcesów")
axes[2].plot(history["epsilon"], color="tab:red", label="epsilon")
axes[2].set(xlabel="numer epizodu [-]", ylabel="wartość [-]", title="Sukcesy i eksploracja")
axes[2].legend()
plt.tight_layout()
plt.show()

## Wyuczona polityka

Wykres pokazuje prędkość wybieraną dla danego błędu. Oczekujemy ruchu w stronę celu i wolniejszych prędkości blisko zera. Szare punkty oznaczają nieodwiedzone stany.

In [ ]:
def greedy_policy(q_table):
    """Najlepsza prędkość [m/s] dla każdego stanu oraz maska stanów odwiedzonych."""
    visited = np.any(q_table != 0, axis=1)
    best_idx = np.array([np.argmax(row) for row in q_table])
    return SPEEDS[best_idx], visited


centers = np.array([sum(bin_edges(s)) / 2 for s in range(N_BINS)])
policy_speed, visited = greedy_policy(q_table)

fig, ax = plt.subplots(figsize=(10, 4))
ax.step(centers[visited], policy_speed[visited], where="mid", color="tab:blue", alpha=0.4)
ax.plot(centers[visited], policy_speed[visited], "o", color="tab:blue", label="wybrana prędkość")
ax.plot(centers[~visited], policy_speed[~visited], "o", color="lightgray", label="stan nieodwiedzony")
ax.axvspan(-TOLERANCE, TOLERANCE, color="green", alpha=0.2, label="strefa sukcesu")
ax.set(xlabel="błąd e = x_cel - x_robot [m]", ylabel="wybrana prędkość v [m/s]",
       title="Polityka: prędkość w zależności od odległości do celu")
ax.legend()
plt.show()

fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(q_table.T, aspect="auto", cmap="viridis", origin="lower",
               extent=[E_MIN, E_MAX, -0.5, len(SPEEDS) - 0.5])
ax.set_yticks(range(len(SPEEDS)))
ax.set_yticklabels([f"{v:+.1f}" for v in SPEEDS])
ax.set(xlabel="błąd e [m] (przedział stanu)", ylabel="prędkość v [m/s]", title="Tablica Q")
fig.colorbar(im, label="Q [-]")
plt.show()

## Demonstracja

Wykresy pokazują dojazd do celu po prawej i po lewej stronie. Obserwuj, czy błąd zbliża się do zera i czy robot mieści się w pasie tolerancji.

In [ ]:
def plot_episode(trace, title):
    t = np.array(trace["t"])
    fig, axes = plt.subplots(1, 3, figsize=(16, 3.8))
    axes[0].plot(t, trace["x"], label="x_robot")
    axes[0].plot(t, trace["x_goal"], "r--", label="x_cel")
    axes[0].set(xlabel="czas t [s]", ylabel="pozycja [m]", title=f"{title}: pozycja")
    axes[0].legend()

    axes[1].plot(t, trace["e"], color="tab:orange", label="e(t)")
    axes[1].axhspan(-TOLERANCE, TOLERANCE, color="green", alpha=0.2, label="tolerancja")
    axes[1].axhline(0, color="k", lw=0.5)
    axes[1].set(xlabel="czas t [s]", ylabel="błąd e [m]", title=f"{title}: błąd")
    axes[1].legend()

    axes[2].step(t[1:], trace["v"], where="pre", color="tab:green", label="v(t)")
    axes[2].set(xlabel="czas t [s]", ylabel="prędkość v [m/s]", title=f"{title}: akcje")
    axes[2].legend()
    plt.tight_layout()
    plt.show()


rng_demo = np.random.default_rng(SEED)
demo_cases = {"cel po prawej": (-3.0, 3.0), "cel po lewej": (3.0, -3.0)}
traces = {}
for name, (x0, xg) in demo_cases.items():
    trace, ok = run_greedy_episode(env, encode, q_table, MAX_STEPS, rng_demo, x0, xg)
    traces[name] = trace
    print(f"{name}: start x={x0:+.1f} m, cel x={xg:+.1f} m -> "
          f"{'sukces' if ok else 'brak sukcesu'} po {len(trace['v'])} krokach ({trace['t'][-1]:.1f} s), "
          f"końcowy błąd {trace['e'][-1]:+.3f} m")
    plot_episode(trace, name)